> # Databricks RAG Copilot for Data Engineering Runbooks

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) solution using Databricks Mosaic AI to enable intelligent querying over enterprise runbook documentation stored as PDF files.

The system ingests a centralized runbook library, parses and structures the content, indexes it using Vector Search, and allows users to ask natural language questions to retrieve troubleshooting steps, validation queries, and resolution guidance.

> **Solution Architecture**

1. PDF Runbooks (Databricks Volume)  
         
2. Document Parsing (ai_parse_document)  
 
3. Text Extraction & Cleaning  

4. Chunking (Section-Based Splitting)  
 
5. Delta Table (Structured Knowledge Base)  
  
6. Mosaic AI Vector Search  

7. RAG (Retrieval + LLM Answer Generation)

In [0]:
#Validate access to rag_volume
display(dbutils.fs.ls("/Volumes/workspace/rag_demo/rag_volume/"))

In [0]:
#Read the PDF from Volume
file_path = "/Volumes/workspace/rag_demo/rag_volume/Enterprise_Runbook_Library.pdf"

In [0]:
#Parse the PDF

#ai_parse_document is a Databricks native AI function designed to extract structured, usable text from documents like PDFs, Word files, and images. It’s part of the Databricks AI Functions ecosystem

df = spark.sql(f"""
SELECT 
  '{file_path}' AS file_path,
  ai_parse_document('{file_path}') AS parsed
""")

display(df)

In [0]:
#Extract text
parsed_string_df = df.selectExpr(
    "file_path",
    "CAST(parsed AS STRING) as parsed_str"
)

display(parsed_string_df)

In [0]:
import re

row = parsed_string_df.collect()[0]
file_path = row["file_path"]
parsed_str = row["parsed_str"]

contents = re.findall(r'"content":"(.*?)"', parsed_str)

print(f"Number of extracted content blocks: {len(contents)}")
print(contents[:10])

In [0]:
clean_text = clean_text.replace('\\"', '"').replace("\\n", "\n").replace("\\t", " ")
print(clean_text[:3000])

In [0]:
clean_doc_df = spark.createDataFrame(
    [(file_path, "Enterprise_Runbook_Library.pdf", clean_text)],
    ["file_path", "file_name", "text"]
)

display(clean_doc_df)

In [0]:
#Save as Delta Table
clean_doc_df.write.mode("overwrite").saveAsTable("workspace.rag_demo.parsed_docs_clean")

In [0]:
#Chunk the document
from pyspark.sql.functions import explode, split

chunk_df = parsed_df.withColumn(
    "chunk",
    explode(split("text", "\n\n"))   # split by paragraphs
).filter("length(chunk) > 50")

display(chunk_df)

In [0]:
#Test Retrieval
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

index = client.get_index(
    endpoint_name="rag-endpoint",
    index_name="workspace.rag_demo.rag_chunks_index"
)

results = index.similarity_search(
    query_text="How to fix pipeline authentication failure?",
    columns=["chunk"],
    num_results=3
)

print(results)